In [70]:
import pandas as pd

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [71]:
#Feature Enginering
train_df["familySize"] = train_df["Parch"] + train_df["SibSp"] + 1
test_df["familySize"] = test_df["Parch"] + test_df["SibSp"] + 1


train_df["isAlone"] = (train_df["familySize"] == 1).astype(int)
test_df["isAlone"] = (test_df["familySize"] == 1).astype(int)

In [72]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
 12  familySize   891 non-null    int64  
 13  isAlone      891 non-null    int32  
dtypes: float64(2), int32(1), int64(6), object(5)
memory usage: 94.1+ KB


In [73]:
#Data cleaning
from sklearn.impute import SimpleImputer

#num imputing
num_impute = SimpleImputer(strategy= "median")
train_df["Age"] = num_impute.fit_transform(train_df[["Age"]])
test_df["Age"] = num_impute.transform(test_df[["Age"]])

#categorial imputing

cat_imputer = SimpleImputer(strategy="most_frequent")
train_df["Embarked"] = cat_imputer.fit_transform(train_df[["Embarked"]]).ravel()
test_df["Embarked"] = cat_imputer.transform(test_df[["Embarked"]]).ravel()

In [74]:
#Fare imputing
fare_impute = SimpleImputer(strategy= "median")
fare_impute.fit(train_df[["Fare"]])
test_df["Fare"] = fare_impute.transform(test_df[["Fare"]])

In [75]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     891 non-null    object 
 12  familySize   891 non-null    int64  
 13  isAlone      891 non-null    int32  
dtypes: float64(2), int32(1), int64(6), object(5)
memory usage: 94.1+ KB


In [76]:
#Feature deletion
train_df = train_df.drop(["Name" , "Cabin" , "Ticket" , "PassengerId"] , axis= 1)
test_df = test_df.drop(["Name" , "Cabin" , "Ticket" , "PassengerId"] , axis= 1)

In [77]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Survived    891 non-null    int64  
 1   Pclass      891 non-null    int64  
 2   Sex         891 non-null    object 
 3   Age         891 non-null    float64
 4   SibSp       891 non-null    int64  
 5   Parch       891 non-null    int64  
 6   Fare        891 non-null    float64
 7   Embarked    891 non-null    object 
 8   familySize  891 non-null    int64  
 9   isAlone     891 non-null    int32  
dtypes: float64(2), int32(1), int64(5), object(2)
memory usage: 66.3+ KB


In [78]:
#Encoding
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

encoder = LabelEncoder()
on_encoder = OneHotEncoder(sparse_output= False , drop='first')
train_df["Sex"] = encoder.fit_transform(train_df["Sex"])
test_df["Sex"] = encoder.transform(test_df["Sex"])

encoder_1 = LabelEncoder()
train_df["Embarked"] = encoder_1.fit_transform(train_df["Embarked"])
test_df["Embarked"] = encoder_1.transform(test_df["Embarked"])

In [79]:
x = train_df.drop("Survived", axis= 1)
y = train_df["Survived"]

In [80]:
#train-test split
from sklearn.model_selection import train_test_split

x_train , x_test , y_train , y_test = train_test_split(x , y , test_size=0.2 , random_state= 42)

In [82]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(random_state= 42)
rf.fit(x_train , y_train)

RandomForestClassifier(random_state=42)

In [85]:
y_pred = rf.predict(x_test)
accuracy_score(y_test , y_pred)

0.8156424581005587

In [89]:
from sklearn.model_selection import cross_val_score

cross = cross_val_score(rf , x_train , y_train , cv = 3 , scoring="accuracy" , n_jobs= -1)
print(f"The accuracy is {cross.mean()}")

The accuracy is 0.7907255729295938


In [91]:
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

In [92]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(rf, param_dist, cv= 5 , n_jobs= -1 , scoring='accuracy')
random_search.fit(x_train , y_train)

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
                   n_jobs=-1,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': [3, 5, 7, 10, 15, None],
                                        'max_features': ['sqrt', 'log2', None],
                                        'min_samples_leaf': [1, 2, 4, 8],
                                        'min_samples_split': [2, 5, 10, 20],
                                        'n_estimators': [100, 200, 300, 500]},
                   scoring='accuracy')

In [93]:
y_pred_1 = random_search.predict(x_test)
accuracy_score(y_test , y_pred_1)

0.8100558659217877